In [ ]:
  # This mounts your Google Drive to the Colab VM.
from google.colab import drive
drive.mount('/content/drive')

# TODO: Enter the foldername in your Drive where you have saved the unzipped
# assignment folder, e.g. 'cs231n/assignments/assignment2/'
FOLDERNAME ="AI_study/CS231n 2017/Assignment2/"
assert FOLDERNAME is not None, "[!] Enter the foldername."

# Now that we've mounted your Drive, this ensures that
# the Python interpreter of the Colab VM can load
# python files from within it.
import sys
sys.path.append('/content/drive/My Drive/{}'.format(FOLDERNAME))

# This downloads the CIFAR-10 dataset to your Drive
# if it doesn't already exist.
%cd /content/drive/My\ Drive/$FOLDERNAME/cs231n/datasets/
!bash get_datasets.sh
%cd /content/drive/My\ Drive/$FOLDERNAME

import sys
import importlib
sys.modules['imp'] = importlib  # imp 대체

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torch.utils.data import sampler
import torch.nn.functional as F

import torchvision.datasets as dset
import torchvision.transforms as T

import numpy as np

USE_GPU = True
dtype = torch.float32 # We will be using float throughout this tutorial.

if USE_GPU and torch.cuda.is_available():
    device = torch.device('cuda')
else:
    device = torch.device('cpu')

# Constant to control how frequently we print train loss.
print_every = 100
print('using device:', device)

NUM_TRAIN = 49000

# The torchvision.transforms package provides tools for preprocessing data
# and for performing data augmentation; here we set up a transform to
# preprocess the data by subtracting the mean RGB value and dividing by the
# standard deviation of each RGB value; we've hardcoded the mean and std.
transform = T.Compose([
                T.ToTensor(),
                T.Normalize((0.4914, 0.4822, 0.4465), (0.2023, 0.1994, 0.2010))
            ])

# We set up a Dataset object for each split (train / val / test); Datasets load
# training examples one at a time, so we wrap each Dataset in a DataLoader which
# iterates through the Dataset and forms minibatches. We divide the CIFAR-10
# training set into train and val sets by passing a Sampler object to the
# DataLoader telling how it should sample from the underlying Dataset.
cifar10_train = dset.CIFAR10('./cs231n/datasets', train=True, download=True,
                             transform=transform)
loader_train = DataLoader(cifar10_train, batch_size=64,
                          sampler=sampler.SubsetRandomSampler(range(NUM_TRAIN)))

cifar10_val = dset.CIFAR10('./cs231n/datasets', train=True, download=True,
                           transform=transform)
loader_val = DataLoader(cifar10_val, batch_size=64,
                        sampler=sampler.SubsetRandomSampler(range(NUM_TRAIN, 50000)))

cifar10_test = dset.CIFAR10('./cs231n/datasets', train=False, download=True,
                            transform=transform)
loader_test = DataLoader(cifar10_test, batch_size=64)

In [ ]:
from tqdm import tqdm
def get_accuracy(loader, model):
    model.eval()
    num_correct, num_samples = 0, 0
    with torch.no_grad():
        for x, y in loader:
            x = x.to(device=device, dtype=dtype)
            y = y.to(device=device, dtype=torch.long)
            _, preds = model(x).max(1)
            num_correct += (preds == y).sum().item()
            num_samples += preds.size(0)
    return 100 * num_correct / num_samples
def train_part34(model, optimizer, scheduler=None, epochs=1, save_every=10):
    model = model.to(device=device)
    history = {'train_loss': [], 'val_acc': []}

    for e in range(epochs):
        model.train()
        running_loss = 0.0

        # ✅ tqdm으로 iteration 진행바
        pbar = tqdm(loader_train, desc=f'Epoch {e+1}/{epochs}', leave=False)
        for x, y in pbar:
            x = x.to(device=device, dtype=dtype)
            y = y.to(device=device, dtype=torch.long)
            scores = model(x)
            loss = F.cross_entropy(scores, y)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            running_loss += loss.item()

            # ✅ iteration마다 loss 실시간 표시
            pbar.set_postfix({'loss': f'{loss.item():.4f}'})

        if scheduler is not None:
            scheduler.step()

        val_acc = get_accuracy(loader_val, model)
        avg_loss = running_loss / len(loader_train)
        history['train_loss'].append(avg_loss)
        history['val_acc'].append(val_acc)

        # ✅ flush=True로 즉시 출력
        print(f'[Epoch {e+1}/{epochs}] loss: {avg_loss:.4f} | val_acc: {val_acc:.2f}% | lr: {optimizer.param_groups[0]["lr"]:.5f}', flush=True)

        if (e + 1) % save_every == 0:
            torch.save({
                'epoch': e + 1,
                'model_state_dict': model.state_dict(),
                'optimizer_state_dict': optimizer.state_dict(),
                'scheduler_state_dict': scheduler.state_dict() if scheduler else None,
                'history': history,
            }, f'/content/drive/MyDrive/resnet110_epoch{e+1}.pth')
            print(f'💾 Saved checkpoint at epoch {e+1}', flush=True)

    return history

In [ ]:
def get_accuracy(loader, model):
    model.eval()
    num_correct, num_samples = 0, 0
    with torch.no_grad():
        for x, y in loader:
            x = x.to(device=device, dtype=dtype)
            y = y.to(device=device, dtype=torch.long)
            _, preds = model(x).max(1)
            num_correct += (preds == y).sum().item()
            num_samples += preds.size(0)
    return 100 * num_correct / num_samples

In [ ]:
class ResBlock(nn.Module):
    def __init__(self, in_channel, out_channel, down_sample=False):
        super().__init__()
        stride = 2 if down_sample else 1

        self.bn1   = nn.BatchNorm2d(in_channel)
        self.relu  = nn.ReLU(inplace=True)
        self.conv1 = nn.Conv2d(in_channel, out_channel, 3, stride, 1, bias=False)

        self.bn2   = nn.BatchNorm2d(out_channel)
        self.conv2 = nn.Conv2d(out_channel, out_channel, 3, 1, 1, bias=False)

        self.pro_cut = (in_channel != out_channel or down_sample)
        if self.pro_cut:
            self.conv3 = nn.Conv2d(in_channel, out_channel, 1, stride, 0, bias=False)

    def forward(self, x):
        # Pre-activation: BN → ReLU → Conv
        out = self.conv1(self.relu(self.bn1(x)))
        out = self.conv2(self.relu(self.bn2(out)))

        shortcut = self.conv3(x) if self.pro_cut else x

        return out + shortcut


resnet110_update_model = nn.Sequential(
    nn.Conv2d(3, 16, 3, 1, 1, bias=False),

    #16*18
    ResBlock(16,16),ResBlock(16,16),ResBlock(16,16),ResBlock(16,16),ResBlock(16,16),ResBlock(16,16),
    ResBlock(16,16),ResBlock(16,16),ResBlock(16,16),ResBlock(16,16),ResBlock(16,16),ResBlock(16,16),
    ResBlock(16,16),ResBlock(16,16),ResBlock(16,16),ResBlock(16,16),ResBlock(16,16),ResBlock(16,16),

    #32*18
    ResBlock(16,32,True),ResBlock(32,32),ResBlock(32,32),ResBlock(32,32),ResBlock(32,32),ResBlock(32,32),
    ResBlock(32,32),ResBlock(32,32),ResBlock(32,32),ResBlock(32,32),ResBlock(32,32),ResBlock(32,32),
    ResBlock(32,32),ResBlock(32,32),ResBlock(32,32),ResBlock(32,32),ResBlock(32,32),ResBlock(32,32),

    #64*18
    ResBlock(32,64,True),ResBlock(64,64),ResBlock(64,64),ResBlock(64,64),ResBlock(64,64),ResBlock(64,64),
    ResBlock(64,64),ResBlock(64,64),ResBlock(64,64),ResBlock(64,64),ResBlock(64,64),ResBlock(64,64),
    ResBlock(64,64),ResBlock(64,64),ResBlock(64,64),ResBlock(64,64),ResBlock(64,64),ResBlock(64,64),

    nn.BatchNorm2d(64),
    nn.ReLU(inplace=True),
    nn.AdaptiveAvgPool2d((1,1)), # 파라미터 감소
    nn.Flatten(),
    nn.Linear(64, 10)
)

learning_rate = 1e-2

optimizer = optim.SGD(resnet110_update_model.parameters(),
                      lr=learning_rate, momentum=0.9,
                      nesterov=True, weight_decay=1e-4)  # weight_decay 추가 권장

scheduler = optim.lr_scheduler.MultiStepLR(
    optimizer,
    milestones=[82, 123],
    gamma=0.1
)

#history = train_part34(resnet110_update_model, optimizer, scheduler=scheduler, epochs=164)

In [ ]:
def check_accuracy_part34(loader, model):
    if loader.dataset.train:
        print('Checking accuracy on validation set')
    else:
        print('Checking accuracy on test set')
    num_correct = 0
    num_samples = 0
    model.eval()  # set model to evaluation mode
    with torch.no_grad():
        for x, y in loader:
            x = x.to(device=device, dtype=dtype)  # move to device, e.g. GPU
            y = y.to(device=device, dtype=torch.long)
            scores = model(x)
            _, preds = scores.max(1)
            num_correct += (preds == y).sum()
            num_samples += preds.size(0)
        acc = float(num_correct) / num_samples
        print('Got %d / %d correct (%.2f)' % (num_correct, num_samples, 100 * acc))

In [ ]:
# 체크포인트 불러오기
checkpoint = torch.load('/content/drive/MyDrive/resnet110_epoch160.pth')

resnet110_update_model.load_state_dict(checkpoint['model_state_dict'])
optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
scheduler.load_state_dict(checkpoint['scheduler_state_dict'])

start_epoch = checkpoint['epoch']  # 80

# 이어서 학습
train_part34(resnet110_update_model, optimizer, scheduler,
             epochs=164 - start_epoch)  # 남은 epoch만

In [ ]:
import torch
import matplotlib.pyplot as plt

checkpoint = torch.load('/content/drive/MyDrive/resnet110_epoch160.pth')
history = checkpoint['history']

test_acc = get_accuracy(loader_test, resnet110_update_model)
val_acc_final = history['val_acc'][-1]

print(f'Val  Accuracy: {val_acc_final:.2f}%')
print(f'Test Accuracy: {test_acc:.2f}%')

fig, axes = plt.subplots(1, 3, figsize=(18, 4))

# 1. Train Loss
axes[0].plot(history['train_loss'], color='blue')
axes[0].set_title('Train Loss')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')

# 2. Val vs Test Accuracy
axes[1].plot(history['val_acc'], color='orange', label='Val Acc')
axes[1].axhline(y=test_acc, color='green', linestyle='-', linewidth=2, label=f'Test Acc ({test_acc:.2f}%)')
axes[1].axvline(x=82,  color='red', linestyle='--', label='lr ×0.1')
axes[1].axvline(x=123, color='red', linestyle='--')
axes[1].set_title('Val vs Test Accuracy')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Accuracy (%)')
axes[1].legend()

# 3. Loss + Val Acc 동시 (이중 축)
ax3 = axes[2]
ax3_twin = ax3.twinx()
ax3.plot(history['train_loss'], color='blue',   label='Train Loss')
ax3_twin.plot(history['val_acc'], color='orange', label='Val Acc')
ax3.set_title('Loss & Val Acc')
ax3.set_xlabel('Epoch')
ax3.set_ylabel('Loss',         color='blue')
ax3_twin.set_ylabel('Acc (%)', color='orange')
ax3.axvline(x=82,  color='red', linestyle='--')
ax3.axvline(x=123, color='red', linestyle='--')
ax3.legend(loc='upper left')
ax3_twin.legend(loc='upper right')

plt.tight_layout()
plt.show()

In [1]:
k = 'abdfdfssf, dfdsf,ef'
set(k)

{' ', ',', 'a', 'b', 'd', 'e', 'f', 's'}

In [ ]:
"""
Minimal character-level Vanilla RNN model. Written by Andrej Karpathy (@karpathy)
BSD License
"""
import numpy as np

# data I/O
data = open('input.txt', 'r').read() # should be simple plain text file
chars = list(set(data))
data_size, vocab_size = len(data), len(chars)
print 'data has %d characters, %d unique.' % (data_size, vocab_size)
char_to_ix = { ch:i for i,ch in enumerate(chars) }
ix_to_char = { i:ch for i,ch in enumerate(chars) }

# hyperparameters
hidden_size = 100 # size of hidden layer of neurons
seq_length = 25 # number of steps to unroll the RNN for
learning_rate = 1e-1

# model parameters
Wxh = np.random.randn(hidden_size, vocab_size)*0.01 # input to hidden
Whh = np.random.randn(hidden_size, hidden_size)*0.01 # hidden to hidden
Why = np.random.randn(vocab_size, hidden_size)*0.01 # hidden to output
bh = np.zeros((hidden_size, 1)) # hidden bias
by = np.zeros((vocab_size, 1)) # output bias

def lossFun(inputs, targets, hprev):
  """
  inputs,targets are both list of integers.
  hprev is Hx1 array of initial hidden state
  returns the loss, gradients on model parameters, and last hidden state
  """
  xs, hs, ys, ps = {}, {}, {}, {}
  hs[-1] = np.copy(hprev)
  loss = 0
  # forward pass
  for t in xrange(len(inputs)):
    xs[t] = np.zeros((vocab_size,1)) # encode in 1-of-k representation
    xs[t][inputs[t]] = 1
    hs[t] = np.tanh(np.dot(Wxh, xs[t]) + np.dot(Whh, hs[t-1]) + bh) # hidden state
    ys[t] = np.dot(Why, hs[t]) + by # unnormalized log probabilities for next chars
    ps[t] = np.exp(ys[t]) / np.sum(np.exp(ys[t])) # probabilities for next chars
    loss += -np.log(ps[t][targets[t],0]) # softmax (cross-entropy loss)
  # backward pass: compute gradients going backwards
  dWxh, dWhh, dWhy = np.zeros_like(Wxh), np.zeros_like(Whh), np.zeros_like(Why)
  dbh, dby = np.zeros_like(bh), np.zeros_like(by)
  dhnext = np.zeros_like(hs[0])
  for t in reversed(xrange(len(inputs))):
    dy = np.copy(ps[t])
    dy[targets[t]] -= 1 # backprop into y. see http://cs231n.github.io/neural-networks-case-study/#grad if confused here
    dWhy += np.dot(dy, hs[t].T)
    dby += dy
    dh = np.dot(Why.T, dy) + dhnext # backprop into h
    dhraw = (1 - hs[t] * hs[t]) * dh # backprop through tanh nonlinearity
    dbh += dhraw
    dWxh += np.dot(dhraw, xs[t].T)
    dWhh += np.dot(dhraw, hs[t-1].T)
    dhnext = np.dot(Whh.T, dhraw)
  for dparam in [dWxh, dWhh, dWhy, dbh, dby]:
    np.clip(dparam, -5, 5, out=dparam) # clip to mitigate exploding gradients
  return loss, dWxh, dWhh, dWhy, dbh, dby, hs[len(inputs)-1]

def sample(h, seed_ix, n):
  """
  sample a sequence of integers from the model
  h is memory state, seed_ix is seed letter for first time step
  """
  x = np.zeros((vocab_size, 1))
  x[seed_ix] = 1
  ixes = []
  for t in xrange(n):
    h = np.tanh(np.dot(Wxh, x) + np.dot(Whh, h) + bh)
    y = np.dot(Why, h) + by
    p = np.exp(y) / np.sum(np.exp(y))
    ix = np.random.choice(range(vocab_size), p=p.ravel())
    x = np.zeros((vocab_size, 1))
    x[ix] = 1
    ixes.append(ix)
  return ixes

n, p = 0, 0
mWxh, mWhh, mWhy = np.zeros_like(Wxh), np.zeros_like(Whh), np.zeros_like(Why)
mbh, mby = np.zeros_like(bh), np.zeros_like(by) # memory variables for Adagrad
smooth_loss = -np.log(1.0/vocab_size)*seq_length # loss at iteration 0
while True:
  # prepare inputs (we're sweeping from left to right in steps seq_length long)
  if p+seq_length+1 >= len(data) or n == 0:
    hprev = np.zeros((hidden_size,1)) # reset RNN memory
    p = 0 # go from start of data
  inputs = [char_to_ix[ch] for ch in data[p:p+seq_length]]
  targets = [char_to_ix[ch] for ch in data[p+1:p+seq_length+1]]

  # sample from the model now and then
  if n % 100 == 0:
    sample_ix = sample(hprev, inputs[0], 200)
    txt = ''.join(ix_to_char[ix] for ix in sample_ix)
    print '----\n %s \n----' % (txt, )

  # forward seq_length characters through the net and fetch gradient
  loss, dWxh, dWhh, dWhy, dbh, dby, hprev = lossFun(inputs, targets, hprev)
  smooth_loss = smooth_loss * 0.999 + loss * 0.001
  if n % 100 == 0: print 'iter %d, loss: %f' % (n, smooth_loss) # print progress

  # perform parameter update with Adagrad
  for param, dparam, mem in zip([Wxh, Whh, Why, bh, by],
                                [dWxh, dWhh, dWhy, dbh, dby],
                                [mWxh, mWhh, mWhy, mbh, mby]):
    mem += dparam * dparam
    param += -learning_rate * dparam / np.sqrt(mem + 1e-8) # adagrad update

  p += seq_length # move data pointer
  n += 1 # iteration counter